# NVIDIA CUDA — V2 benchmark validation (Tesla T4 / Kaggle)

This notebook **re-runs your exact V2 CUDA kernels**. It does not invent results or modify their arithmetic. It checks reproducibility on the same GPU, input-seed sensitivity, and performance at larger `N`.

**Run in Kaggle with GPU enabled.** Allow several minutes. Timing excludes CPU↔GPU transfers and allocations; it is **kernel-only** timing. Kaggle GPU utilization/noisy neighbors can influence timing. The final report is generated from *your actual runs*.

**Pass criteria:** correctness PASS on every run, the cuBLAS setting is reported, and a stable distribution of timings. Near-cuBLAS throughput alone does **not** prove similar performance across all sizes.

In [ ]:
import os, subprocess, pathlib, pandas as pd, numpy as np, matplotlib.pyplot as plt, json
from IPython.display import display
outdir = pathlib.Path('/kaggle/working/cuda_validation') if pathlib.Path('/kaggle/working').exists() else pathlib.Path('cuda_validation')
outdir.mkdir(parents=True, exist_ok=True)
print(subprocess.run(['nvidia-smi'],capture_output=True,text=True).stdout[:5000])
print(subprocess.run(['nvcc','--version'],capture_output=True,text=True).stdout)


## 1. Exact V2 implementation (embedded for reproducibility)

In [ ]:
source = '#include <cuda_runtime.h>\n#include <cublas_v2.h>\n#include <algorithm>\n#include <cmath>\n#include <cstdio>\n#include <cstdlib>\n#include <fstream>\n#include <functional>\n#include <numeric>\n#include <random>\n#include <string>\n#include <vector>\n\n#define CUDA_CHECK(x) do {cudaError_t e=(x);if(e!=cudaSuccess){fprintf(stderr,"CUDA error at %s:%d: %s\\n",__FILE__,__LINE__,cudaGetErrorString(e));exit(1);}}while(0)\n#define BLAS_CHECK(x) do {cublasStatus_t e=(x);if(e!=CUBLAS_STATUS_SUCCESS){fprintf(stderr,"cuBLAS error at %s:%d: %d\\n",__FILE__,__LINE__,int(e));exit(1);}}while(0)\n\n// Row-major C = A x B. Timing excludes allocations and host transfers.\n__global__ void naive_sgemm(const float* A,const float* B,float* C,int N){\n int row=blockIdx.y*blockDim.y+threadIdx.y,col=blockIdx.x*blockDim.x+threadIdx.x;\n if(row>=N||col>=N)return;\n float acc=0.0f;\n for(int k=0;k<N;k++) acc=fmaf(A[row*N+k],B[k*N+col],acc);\n C[row*N+col]=acc;\n}\n\ntemplate<int BM,int BK=16>\n__global__ void register_tiled_sgemm(const float* __restrict__ A,const float* __restrict__ B,float* __restrict__ C,int N){\n static_assert(BM==32||BM==64,"Supported output tiles: 32 or 64");\n constexpr int THREADS=256, TM=BM/16;\n __shared__ float As[BM][BK+1];\n __shared__ float Bs[BK][BM+1];\n int tid=threadIdx.y*16+threadIdx.x;\n int row0=blockIdx.y*BM+threadIdx.y*TM;\n int col0=blockIdx.x*BM+threadIdx.x*TM;\n float acc[TM][TM]={};\n for(int kb=0;kb<N;kb+=BK){\n   for(int idx=tid;idx<BM*BK;idx+=THREADS){\n     int r=idx/BK,k=idx%BK;\n     int gr=blockIdx.y*BM+r,gk=kb+k;\n     As[r][k]=(gr<N&&gk<N)?A[gr*N+gk]:0.0f;\n   }\n   for(int idx=tid;idx<BK*BM;idx+=THREADS){\n     int k=idx/BM,c=idx%BM;\n     int gk=kb+k,gc=blockIdx.x*BM+c;\n     Bs[k][c]=(gk<N&&gc<N)?B[gk*N+gc]:0.0f;\n   }\n   __syncthreads();\n   #pragma unroll\n   for(int k=0;k<BK;k++){\n     float a[TM],b[TM];\n     #pragma unroll\n     for(int i=0;i<TM;i++)a[i]=As[threadIdx.y*TM+i][k];\n     #pragma unroll\n     for(int j=0;j<TM;j++)b[j]=Bs[k][threadIdx.x*TM+j];\n     #pragma unroll\n     for(int i=0;i<TM;i++){\n       #pragma unroll\n       for(int j=0;j<TM;j++)acc[i][j]=fmaf(a[i],b[j],acc[i][j]);\n     }\n   }\n   __syncthreads();\n }\n #pragma unroll\n for(int i=0;i<TM;i++){\n   #pragma unroll\n   for(int j=0;j<TM;j++){\n     int row=row0+i,col=col0+j;\n     if(row<N&&col<N)C[row*N+col]=acc[i][j];\n   }\n }\n}\n\n__global__ void reduce_shared(const float* in,float* out,int n){\n extern __shared__ float s[];\n int t=threadIdx.x,i=blockIdx.x*(blockDim.x*2)+t;\n float x=(i<n?in[i]:0.0f)+(i+blockDim.x<n?in[i+blockDim.x]:0.0f);\n s[t]=x;__syncthreads();\n for(int step=blockDim.x/2;step>0;step>>=1){if(t<step)s[t]+=s[t+step];__syncthreads();}\n if(t==0)out[blockIdx.x]=s[0];\n}\n__global__ void reduce_shuffle(const float* in,float* out,int n){\n __shared__ float warp[8];\n int t=threadIdx.x,i=blockIdx.x*(blockDim.x*2)+t;\n float x=(i<n?in[i]:0.0f)+(i+blockDim.x<n?in[i+blockDim.x]:0.0f);\n for(int offset=16;offset>0;offset/=2)x+=__shfl_down_sync(0xffffffff,x,offset);\n if((t&31)==0)warp[t/32]=x;\n __syncthreads();\n if(t<32){x=t<8?warp[t]:0.0f;\n   for(int offset=16;offset>0;offset/=2)x+=__shfl_down_sync(0xffffffff,x,offset);\n   if(t==0)out[blockIdx.x]=x;\n }\n}\n\nfloat median_ms(const std::function<void()>& fn,int repeats=7,int launches=10){\n cudaEvent_t start,stop;CUDA_CHECK(cudaEventCreate(&start));CUDA_CHECK(cudaEventCreate(&stop));\n for(int i=0;i<3;i++)fn();CUDA_CHECK(cudaGetLastError());CUDA_CHECK(cudaDeviceSynchronize());\n std::vector<float> timings;\n for(int r=0;r<repeats;r++){\n   CUDA_CHECK(cudaEventRecord(start));\n   for(int j=0;j<launches;j++)fn();\n   CUDA_CHECK(cudaGetLastError());CUDA_CHECK(cudaEventRecord(stop));CUDA_CHECK(cudaEventSynchronize(stop));\n   float ms;CUDA_CHECK(cudaEventElapsedTime(&ms,start,stop));timings.push_back(ms/launches);\n }\n std::sort(timings.begin(),timings.end());CUDA_CHECK(cudaEventDestroy(start));CUDA_CHECK(cudaEventDestroy(stop));\n return timings[timings.size()/2];\n}\n\nbool validate(const std::vector<float>& got,const std::vector<float>& ref,float& max_abs,float& max_rel){\n max_abs=0;max_rel=0;int errors=0;\n for(size_t i=0;i<ref.size();i++){\n   if(!std::isfinite(got[i])){errors++;continue;}\n   float diff=std::fabs(got[i]-ref[i]);\n   float rel=diff/std::max(1.0f,std::fabs(ref[i]));\n   max_abs=std::max(diff,max_abs);max_rel=std::max(rel,max_rel);\n   if(diff>0.002f&&rel>0.0005f)errors++;\n }\n return errors==0;\n}\n\nint main(){\n cudaDeviceProp prop{};int dev;CUDA_CHECK(cudaGetDevice(&dev));CUDA_CHECK(cudaGetDeviceProperties(&prop,dev));\n printf("GPU: %s, compute capability %d.%d, SMs %d\\n",prop.name,prop.major,prop.minor,prop.multiProcessorCount);\n if(prop.major<5){fprintf(stderr,"This notebook requires a supported modern CUDA GPU.\\n");return 1;}\n std::ofstream csv("sgemm_v2_results.csv");if(!csv)csv.open("sgemm_v2_results.csv");\n if(!csv){fprintf(stderr,"Cannot write SGEMM CSV\\n");return 1;}\n csv<<"N,kernel,median_ms,gflops,ratio_to_cublas,correct,max_abs,max_rel\\n";\n cublasHandle_t handle;BLAS_CHECK(cublasCreate(&handle));\n // Avoid implicit reduced precision for comparisons where this cuBLAS math mode is supported.\n BLAS_CHECK(cublasSetMathMode(handle,CUBLAS_PEDANTIC_MATH));\n bool all_ok=true;\n std::mt19937 rng(2026);std::uniform_real_distribution<float> dist(-0.5f,0.5f);\n for(int N:{127,256,512,1024}){\n   const size_t count=size_t(N)*N,bytes=count*sizeof(float);\n   std::vector<float> a(count),b(count),ref(count),out(count);\n   for(size_t i=0;i<count;i++){a[i]=dist(rng);b[i]=dist(rng);}\n   float *da,*db,*dc;CUDA_CHECK(cudaMalloc(&da,bytes));CUDA_CHECK(cudaMalloc(&db,bytes));CUDA_CHECK(cudaMalloc(&dc,bytes));\n   CUDA_CHECK(cudaMemcpy(da,a.data(),bytes,cudaMemcpyHostToDevice));\n   CUDA_CHECK(cudaMemcpy(db,b.data(),bytes,cudaMemcpyHostToDevice));\n   const float alpha=1.f,beta=0.f;\n   // cuBLAS uses column-major layout: swap A/B to get row-major A*B.\n   auto blas=[&](){BLAS_CHECK(cublasSgemm(handle,CUBLAS_OP_N,CUBLAS_OP_N,N,N,N,&alpha,db,N,da,N,&beta,dc,N));};\n   float blas_ms=median_ms(blas);\n   CUDA_CHECK(cudaMemcpy(ref.data(),dc,bytes,cudaMemcpyDeviceToHost));\n   printf("\\nN=%d | cuBLAS (pedantic FP32 math): %.5f ms\\n",N,blas_ms);\n   auto run=[&](const char* name,const std::function<void()>& launch){\n     launch();CUDA_CHECK(cudaGetLastError());CUDA_CHECK(cudaDeviceSynchronize());\n     CUDA_CHECK(cudaMemcpy(out.data(),dc,bytes,cudaMemcpyDeviceToHost));\n     float abs_err,rel_err;bool ok=validate(out,ref,abs_err,rel_err);all_ok&=ok;\n     float ms=median_ms(launch);\n     double gflops=2.0*N*N*N/(ms*1e6),ratio=blas_ms/ms;\n     printf(" %-12s %s median %.5f ms %9.1f GFLOPS %5.2fx cuBLAS | abs %.5g rel %.5g\\n",name,ok?"PASS":"FAIL",ms,gflops,ratio,abs_err,rel_err);\n     csv<<N<<\',\'<<name<<\',\'<<ms<<\',\'<<gflops<<\',\'<<ratio<<\',\'<<(ok?1:0)<<\',\'<<abs_err<<\',\'<<rel_err<<\'\\n\';\n   };\n   dim3 block(16,16),naive_grid((N+15)/16,(N+15)/16);\n   run("naive",[&](){naive_sgemm<<<naive_grid,block>>>(da,db,dc,N);});\n   dim3 grid32((N+31)/32,(N+31)/32),grid64((N+63)/64,(N+63)/64);\n   run("tile32_reg2",[&](){register_tiled_sgemm<32><<<grid32,block>>>(da,db,dc,N);});\n   run("tile64_reg4",[&](){register_tiled_sgemm<64><<<grid64,block>>>(da,db,dc,N);});\n   double cugflops=2.0*N*N*N/(blas_ms*1e6);\n   csv<<N<<",cublas,"<<blas_ms<<\',\'<<cugflops<<",1,1,0,0\\n";\n   CUDA_CHECK(cudaFree(da));CUDA_CHECK(cudaFree(db));CUDA_CHECK(cudaFree(dc));\n }\n BLAS_CHECK(cublasDestroy(handle));csv.close();\n std::ofstream rcsv("reduction_v2_results.csv");if(!rcsv)rcsv.open("reduction_v2_results.csv");\n if(!rcsv){fprintf(stderr,"Cannot write reduction CSV\\n");return 1;}\n rcsv<<"N,kernel,median_ms,result,reference,rel_error,correct\\n";\n printf("\\nTWO-STAGE REDUCTION: timings include both stages; transfers excluded\\n");\n constexpr int THREADS=256;\n for(int N:{(1<<20),(1<<22)}){\n   std::vector<float> input(N);double reference=0;\n   for(int i=0;i<N;i++){input[i]=(i%13+1)/13.0f;reference+=input[i];}\n   int blocks=(N+2*THREADS-1)/(2*THREADS);\n   float *di,*p1,*p2;CUDA_CHECK(cudaMalloc(&di,size_t(N)*sizeof(float)));\n   CUDA_CHECK(cudaMalloc(&p1,blocks*sizeof(float)));CUDA_CHECK(cudaMalloc(&p2,blocks*sizeof(float)));\n   CUDA_CHECK(cudaMemcpy(di,input.data(),size_t(N)*sizeof(float),cudaMemcpyHostToDevice));\n   auto run=[&](const char* name,bool use_shuffle){\n      // Reduce repeatedly until exactly one value remains.\n      auto full=[&](){\n        int n=N;const float* src=di;float* dst=p1;\n        while(n>1){\n          int nb=(n+2*THREADS-1)/(2*THREADS);\n          if(use_shuffle)reduce_shuffle<<<nb,THREADS>>>(src,dst,n);\n          else reduce_shared<<<nb,THREADS,THREADS*sizeof(float)>>>(src,dst,n);\n          n=nb;src=dst;dst=(dst==p1?p2:p1);\n        }\n      };\n      full();CUDA_CHECK(cudaGetLastError());CUDA_CHECK(cudaDeviceSynchronize());\n      // The final buffer is determined by the number of reduction passes.\n      int n=N;float* last=nullptr;float* next=p1;\n      while(n>1){int nb=(n+2*THREADS-1)/(2*THREADS);last=next;next=(next==p1?p2:p1);n=nb;}\n      float got;CUDA_CHECK(cudaMemcpy(&got,last,sizeof(float),cudaMemcpyDeviceToHost));\n      double rel=std::fabs(double(got)-reference)/std::max(1.0,std::fabs(reference));\n      bool ok=rel<1e-4;all_ok&=ok;\n      float ms=median_ms(full);\n      printf(" N=%d %-8s %s median %.5f ms relerr %.3g\\n",N,name,ok?"PASS":"FAIL",ms,rel);\n      rcsv<<N<<\',\'<<name<<\',\'<<ms<<\',\'<<got<<\',\'<<reference<<\',\'<<rel<<\',\'<<(ok?1:0)<<\'\\n\';\n   };\n   run("shared",false);run("shuffle",true);\n   CUDA_CHECK(cudaFree(di));CUDA_CHECK(cudaFree(p1));CUDA_CHECK(cudaFree(p2));\n }\n printf("\\nALL CORRECTNESS CHECKS: %s\\n",all_ok?"PASS":"FAIL");\n return all_ok?0:2;\n}\n'
(outdir/'sgemm_v2_original.cu').write_text(source)
print(f'Saved {len(source)} characters of original V2 source')

## 2. Compile once, run independently 3 times

Each process creates fresh random matrices and GPU buffers, and V2 performs its own correctness tests. Multiple process runs can reveal variation that the median of event timings inside one process cannot detect.

In [ ]:
import shutil, time
exe = outdir/'sgemm_v2_validation'
compile_cmd=['nvcc','-O3','-std=c++17',str(outdir/'sgemm_v2_original.cu'),'-lcublas','-o',str(exe)]
print(' '.join(compile_cmd))
r=subprocess.run(compile_cmd,text=True,capture_output=True)
print(r.stdout, r.stderr[-6000:])
assert r.returncode==0, 'CUDA compilation failed; share error output'
print('Compile PASS')

In [ ]:
runs=[]
for i in range(1,4):
    print(f'\n===== INDEPENDENT RUN {i}/3 =====',flush=True)
    run=subprocess.run([str(exe)],text=True,capture_output=True,cwd=outdir)
    (outdir/f'console_run{i}.txt').write_text(run.stdout+'\n'+run.stderr)
    print(run.stdout[-2300:])
    if run.returncode!=0:
        print(run.stderr[-3000:])
        raise RuntimeError(f'Run {i} failed correctness or execution (exit {run.returncode})')
    assert 'ALL CORRECTNESS CHECKS: PASS' in run.stdout
    for kind in ('sgemm','reduction'):
        source_csv=outdir/f'{kind}_v2_results.csv'
        assert source_csv.exists(), f'Missing {source_csv}'
        target=outdir/f'{kind}_run{i}.csv'
        shutil.copyfile(source_csv,target)
    runs.append(i)
print('ALL 3 RUNS: PASS')

## 3. Aggregate repeatability and correctness

In [ ]:
data=[]; red=[]
for i in runs:
    df=pd.read_csv(outdir/f'sgemm_run{i}.csv'); df['run']=i; data.append(df)
    rr=pd.read_csv(outdir/f'reduction_run{i}.csv'); rr['run']=i; red.append(rr)
sg=pd.concat(data,ignore_index=True); reduction=pd.concat(red,ignore_index=True)
assert (sg['correct']==1).all() and (reduction['correct']==1).all(), 'Some correctness tests FAILED'
summary=(sg.groupby(['N','kernel'],as_index=False).agg(
    median_ms=('median_ms','median'), min_ms=('median_ms','min'), max_ms=('median_ms','max'),
    mean_ms=('median_ms','mean'), std_ms=('median_ms','std'), median_gflops=('gflops','median'),
    max_abs=('max_abs','max'), max_rel=('max_rel','max'), trials=('run','nunique')))
summary['range_pct_of_median']=100*(summary['max_ms']-summary['min_ms'])/summary['median_ms']
summary.to_csv(outdir/'sgemm_validation_summary.csv',index=False)
print('SGEMM summary (variation between independent runs):')
display(summary.round(4))
redsum=(reduction.groupby(['N','kernel'],as_index=False).agg(median_ms=('median_ms','median'),max_rel_error=('rel_error','max'),trials=('run','nunique')))
redsum.to_csv(outdir/'reduction_validation_summary.csv',index=False)
print('Reduction summary:'); display(redsum.round(6))


## 4. Plot measured performance and cross-run variation

In [ ]:
for N in sorted(sg.N.unique()):
    subset=sg[sg.N==N]
    piv=subset.pivot(index='run',columns='kernel',values='gflops')
    ax=piv.plot(kind='bar',figsize=(8,3.5),rot=0,title=f'SGEMM throughput at N={N} across independent runs')
    ax.set_ylabel('GFLOPS'); ax.set_xlabel('Run'); plt.tight_layout()
    plt.savefig(outdir/f'sgemm_N{N}_repeatability.png',dpi=140)
    plt.show()


## 5. Optional stress run: N=2048 + alternate random seed

Run this cell after the main validation. This produces a **second executable using the exact same kernels**, adding `N=2048` and changing the input random seed. It takes longer because the naive kernel at N=2048 does substantially more work. Measurements are not directly comparable to other machines/GPUs.

In [ ]:
stress_source = source.replace('std::mt19937 rng(2026)', 'std::mt19937 rng(941)')
stress_source = stress_source.replace('for(int N:{127,256,512,1024})', 'for(int N:{127,256,512,1024,2048})')
assert stress_source != source and 'for(int N:{127,256,512,1024,2048})' in stress_source
stress_path=outdir/'sgemm_v2_stress.cu'; stress_path.write_text(stress_source)
stress_exe=outdir/'sgemm_v2_stress'
r=subprocess.run(['nvcc','-O3','-std=c++17',str(stress_path),'-lcublas','-o',str(stress_exe)],capture_output=True,text=True)
assert r.returncode==0,r.stderr[-4000:]
r=subprocess.run([str(stress_exe)],cwd=outdir,capture_output=True,text=True)
(outdir/'stress_console.txt').write_text(r.stdout+'\n'+r.stderr)
print(r.stdout[-4000:]); print(r.stderr[-1500:]); assert r.returncode==0
shutil.copyfile(outdir/'sgemm_v2_results.csv',outdir/'sgemm_stress_seed941.csv')
shutil.copyfile(outdir/'reduction_v2_results.csv',outdir/'reduction_stress_seed941.csv')
display(pd.read_csv(outdir/'sgemm_stress_seed941.csv').round(4))


## 6. Interpretation rules and export

- **Correctness:** every `correct` value must be `1`. V2 compares custom SGEMM to cuBLAS with tolerance and uses randomized inputs. It does **not** use a high-precision CPU reference; add that for a particularly rigorous mathematical check.
- **Fairness:** source calls `cublasSetMathMode(handle,CUBLAS_PEDANTIC_MATH)` and `cublasSgemm`, using the same input matrices and device-resident outputs. This is a sound FP32 comparison for this GPU, but still record software version and GPU model.
- **Stability:** if cross-process timing ranges are large (e.g. >10%), repeat the experiment on a quieter session. This threshold is a practical heuristic, not a formal guarantee.
- **Across versions:** V1 and V2 timings differ in methodology; prefer comparing V2 kernels **within the same notebook and session**.
- **Resume:** report `N`, GPU model, median timing, baseline speedup and arithmetic mode. Do not describe the project as memory-bound/compute-bound without real counter evidence.

Download the ZIP after it is generated and share `sgemm_validation_summary.csv`, stress CSV, or a screenshot of the tables.

In [ ]:
import zipfile
zip_path=outdir.parent/'cuda_validation_results.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for f in outdir.iterdir():
        if f.suffix in ('.csv','.txt','.png'):
            z.write(f,arcname=f.name)
print('Result ZIP:', zip_path)
